# Hierarchical QAOA for Max-Cut

Both source code stages, the fresh 40-node graph with edge probability 0.2, a six-qubit cap per community, two QAOA layers, and 20 objective queries per community. Signed contractions and original-node flips, finite sample ranking, both classical comparisons and the complete labeled graph. Independent absolute Circuit states and cut tables for all realized quantum subproblems.

Native public COBYLA with a hard 20-query cap uses the exact mean negative cut. The initial schedule follows the norm formula reviewed in current Classiq SDK 1.35.0. The source leaves the historical SDK and shot default unspecified; use 1000 actual CPU samples per quantum subproblem, seed 10, and resolve identical cut/count ties by ascending word. Every trial specializes a fresh native program at synthesis. Keep Python seed 246 and use NumPy seed 4812 for layout with local RNGs. No frozen subproblem, historical trajectory equivalence, runtime parameter binding, hardware or optimum claim.

Run from the repository root or start Jupyter there.

## Quantum subproblems remain distinct from graph contraction

The visible community qfunc prepares up to six decision bits and two weighted
cut/mixer layers. FlagQuantum compiles signed coefficients and named outputs.
The host workflow partitions the original forty-node graph, optimizes each
bounded community and contracts it; it does not simulate forty qubits or ask
the compiler to discover the contraction strategy.

Try changing a nonzero subproblem parameter in quantum.py and inspect its
complete distribution before changing community assignments. Every contraction
must preserve signed edge weights and mapping back to original vertices.
Twenty-query subproblem budgets and finite samples do not certify a maximum
cut; the random and greedy cuts are classical comparisons, not quantum results.


In [ ]:
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    phase,
    qfunc,
)


@qfunc
def community_program(
    node_count: int,
    first: list[int],
    second: list[int],
    weights: list[float],
    params: list[float],
    target: Output[QArray[QBit]],
) -> None:
    allocate(node_count, target)
    hadamard_transform(target)
    for layer in range(2):
        for edge in range(len(weights)):
            phase(
                weights[edge]
                * (
                    target[first[edge]]
                    + target[second[edge]]
                    - 2 * target[first[edge]] * target[second[edge]]
                ),
                params[2 * layer],
            )
        for bit in range(node_count):
            RX(params[2 * layer + 1], target[bit])


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.qaoa_in_qaoa.overview.model import (
    fresh_graph,
    split_communities,
)
from examples.synthesis.applications.optimization.qaoa_in_qaoa.overview.plots import (
    plot_solution,
)
from examples.synthesis.applications.optimization.qaoa_in_qaoa.overview.workflow import (
    run_workflow,
)

graph, rng = fresh_graph()
groups = split_communities(graph, rng)
print(f"Fresh graph: {len(graph)} nodes and {len(graph.edges)} edges")
print("Community sizes:", [len(group) for group in groups])


## Solve and contract

Use the [native quantum function](quantum.py) to phase each community’s signed cut, sample the optimized native circuit, and propagate the sampled flips. Compare the observed cut against two classical heuristics.

In [ ]:
workflow = run_workflow()
print("Contraction sizes:", workflow.contraction_sizes)
for index, result in enumerate(workflow.runs):
    print(
        f"Subproblem {index}: nodes={result.nodes}; "
        f"queries={len(result.trace)}; shots={sum(row.count for row in result.observations)}; "
        f"budget stop={result.budget_reached}; resources={result.resources}"
    )
print("Observed assignment:", dict(enumerate(workflow.assignment)))
print("Observed QAOA cut:", workflow.cut)
print("Greedy one-exchange cut:", workflow.greedy_cut)
print("Random partition cut:", workflow.random_cut)
figure = plot_solution(
    workflow, Path("artifacts/development/qaoa-in-qaoa/observed-cut.png")
)
display(Image(filename=str(figure)))
